In [1]:
# SGD
# sklearn - iris dataset

# Imports
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [45]:
# Load the dataset
iris_data = load_iris()
X, y = iris_data.data, iris_data.target

In [46]:
# Preprocess the data
# FIX: Add sparse_output=False to get dense array
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y.reshape(-1,1))

In [47]:
# split the data
# scale the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [48]:
# initialize parameters
np.random.seed(42)  # reproducibility, give same random numbers
weights = np.random.randn(X_train.shape[1], y_train.shape[1])
bias = np.random.randn(y_train.shape[1])

In [49]:
# define the loss function
def compute_loss(X, y, weights, bias):
    predictions = softmax(np.dot(X, weights) + bias)
    loss = -np.mean(np.sum(y * np.log(predictions + 1e-15), axis=1))  # Added small epsilon to avoid log(0)
    return loss

def softmax(z):
    if z.ndim == 1:
        z = z.reshape(1, -1)
    # FIX: Use keepdims=True (not keepdims=1)
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [50]:
# def -> implement SGD algorithm
def sgd(X, y, weights, bias, learning_rate, epochs):
    for epoch in range(epochs):
        for i in range(X.shape[0]):
            # compute the prediction
            z = np.dot(X[i], weights) + bias
            prediction = softmax(z).flatten()

            # Compute the error
            error = prediction - y[i]

            # update the weights and bias
            weights = weights - learning_rate * np.outer(X[i], error)  # compute the derivative
            bias = bias - learning_rate * error
        if epoch % 10 == 0:
            loss = compute_loss(X, y, weights, bias)
            print(f'Epoch: {epoch}, loss: {loss}')

    return weights, bias

In [51]:
# Train the model
learning_rate = 0.01
epochs = 100

weights, bias = sgd(X_train, y_train, weights, bias, learning_rate, epochs)

Epoch: 0, loss: 1.088030058166198
Epoch: 10, loss: 0.3394229832090371
Epoch: 20, loss: 0.25877241037361487
Epoch: 30, loss: 0.2152833378451718
Epoch: 40, loss: 0.1874891269594485
Epoch: 50, loss: 0.168042435469549
Epoch: 60, loss: 0.153612142396729
Epoch: 70, loss: 0.14244701115914246
Epoch: 80, loss: 0.13353214357163307
Epoch: 90, loss: 0.12623674314524103


In [53]:
# evaluate the model 

def predict(X, weights, bias):
    predictions = softmax(np.dot(X, weights) + bias) 
    return np.argmax(predictions, axis = 1)


y_train_pred = predict(X_train, weights, bias)
y_test_pred  = predict(X_test, weights, bias)

y_train_true = np.argmax(y_train, axis=1)
y_test_true  = np.argmax(y_test, axis=1)

train_acc = accuracy_score(y_train_true, y_train_pred)
test_acc  = accuracy_score(y_test_true, y_test_pred)

print(f'Training Accuracy: {train_acc}')
print(f'Testing Accuracy: {test_acc}')

Training Accuracy: 0.9666666666666667
Testing Accuracy: 1.0
